# 212. Word Search II
**Difficulty:** 🔴 Hard · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/word-search-ii/

## 💡 Concepts

**Core concept(s):** A **trie** of all target words + **backtracking DFS** over the grid.

**Why it applies here:** Searching each word separately re-walks the board over and over. Instead, put all words in a trie and sweep the board once: from each cell, follow the trie while the letters match, collecting any word that completes. The trie prunes dead paths instantly.

**Key intuition:** Walk the grid guided by the trie — only step to a neighbor if some word continues that way.

---

### 📚 What is a Trie (Prefix Tree)?
A **trie** stores words letter-by-letter along paths from a root, so words sharing a prefix share the same early path. A marker flags where a word ends.
- **Complexity:** insert / search a word of length L is **O(L)**, no matter how many words are stored.
- **In Python:** nested `dict`s (`{char: child}`) with a sentinel like `'$'` for word-ends.

### 📚 What is Backtracking?
**Backtracking** makes a choice, explores, then **undoes** it to try another. On a grid: mark a cell used, explore neighbors, then unmark it.
- **Complexity:** exponential in the worst case; pruning dead ends early (e.g. with a trie) makes it practical.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

---

**Prerequisite knowledge:**
- Tries.
- Grid DFS with backtracking (mark a cell used, then restore it).

## 📝 Problem

Given a grid of letters and a list of words, return all words that can be formed by connecting adjacent cells (up/down/left/right), no cell reused within a word.

**Example**
```
board = [["o","a","a","n"],
         ["e","t","a","e"],
         ["i","h","k","r"],
         ["i","f","l","v"]]
words = ["oath","pea","eat","rain"] -> ["oath","eat"]
```

> Two approaches: search each word `O(W·m·n·4^L)` and one trie-guided sweep `O(m·n·4^L)`.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Search Each Word (worst)

**Idea:** For every word, run a grid DFS looking for it. Re-walks the whole board once per word.

**Time:** `O(W · m·n · 4^L)`.

**Space:** `O(L)`.

In [ ]:
def find_words_brute(board: List[List[str]], words: List[str]) -> List[str]:
    rows, cols = len(board), len(board[0])
    def exists(word):                      # can we spell this one word somewhere on the board?
        def dfs(r, c, i):
            if i == len(word):
                return True                # matched every letter -> found it
            if r < 0 or c < 0 or r >= rows or c >= cols or board[r][c] != word[i]:
                return False               # off the board, or letter doesn't match
            tmp = board[r][c]; board[r][c] = "#"   # mark this cell as used
            found = (dfs(r+1,c,i+1) or dfs(r-1,c,i+1) or
                     dfs(r,c+1,i+1) or dfs(r,c-1,i+1))  # try all 4 neighbors for the next letter
            board[r][c] = tmp              # restore the cell (backtrack)
            return found
        return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))  # try every start
    return [w for w in words if exists(w)]  # re-scans the whole board once per word (slow)

### Approach 2 — Trie + One Sweep (optimal)

**Idea:** Build a trie of all words. DFS the board once; at each cell, only continue if the letter is a child in the trie. When a node marks a word's end, collect it.

**Time:** `O(m·n · 4^L)` (one sweep; the trie prunes).

**Space:** `O(total letters of words)`.

In [ ]:
def find_words_trie(board: List[List[str]], words: List[str]) -> List[str]:
    root = {}                              # build a trie of all the words
    for w in words:
        node = root
        for c in w:
            node = node.setdefault(c, {})  # walk/create the letter path
        node["$"] = w                      # store the whole word at its end node
    rows, cols = len(board), len(board[0])
    res = []
    def dfs(r, c, node):                   # walk the board, guided by the trie
        if r < 0 or c < 0 or r >= rows or c >= cols:
            return
        ch = board[r][c]
        if ch == "#" or ch not in node:    # used cell, or no word continues with this letter
            return                         # -> dead end, prune immediately
        nxt = node[ch]
        if "$" in nxt:                     # a word ends here
            res.append(nxt.pop("$"))       # collect it (pop so we don't report duplicates)
        board[r][c] = "#"                  # mark used
        dfs(r+1,c,nxt); dfs(r-1,c,nxt); dfs(r,c+1,nxt); dfs(r,c-1,nxt)  # explore neighbors
        board[r][c] = ch                   # restore (backtrack)
    for r in range(rows):                  # one sweep of the board finds ALL words at once
        for c in range(cols):
            dfs(r, c, root)
    return res

In [ ]:
# Correctness check
board = [["o","a","a","n"],["e","t","a","e"],["i","h","k","r"],["i","f","l","v"]]
words = ["oath","pea","eat","rain"]
a = sorted(find_words_brute([row[:] for row in board], words))
b = sorted(find_words_trie([row[:] for row in board], words))
print("brute:", a, "| trie:", b)
assert a == b == ["eat","oath"], "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on a growing number of words `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

The board is fixed; we grow the number of words. Searching per word re-scans the board each time; the trie sweeps once.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

BOARD = [["o","a","a","n"],["e","t","a","e"],["i","h","k","r"],["i","f","l","v"]]

def _words(n):
    # distinct 3-letter words that mostly aren't on the board -> lots of failed searches
    out = []
    for i in range(n):
        s = ""; x = i
        for _ in range(3):
            s += chr(ord("a") + x % 26); x //= 26
        out.append(s)
    return out

def brute_run(words):
    return find_words_brute([row[:] for row in BOARD], words)

def trie_run(words):
    return find_words_trie([row[:] for row in BOARD], words)

def make_worst_case(n):
    return (_words(n),)

solutions = {
    "per-word  O(W * board)": brute_run,
    "trie-sweep O(board)   ": trie_run,
}
sizes = [500, 1000, 2000, 4000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Trie + grid DFS:** search many words at once by letting the trie steer a single board sweep and prune dead ends.
- **Backtracking on a grid:** mark a cell used, explore, then restore — the universal maze/grid pattern.
- **Signal:** "find many words in a grid", "prefix-guided search", "board + dictionary".
- **Related problems:** Word Search (I), Implement Trie, Number of Islands.
- **Common pitfalls:** (1) not restoring a cell after exploring (breaks other paths); (2) reporting duplicates (pop the word marker once found).